# 1. Data Ingestion & Storage
**AAI-540 MLOps Project — Group 6: Student Performance Prediction**

This notebook handles:
- Loading raw student performance data
- Initial data inspection and quality checks
- Combining Math and Portuguese datasets
- Uploading data to S3

In [ ]:
import pandas as pd
import numpy as np
import os
import sagemaker
import boto3
import warnings
warnings.filterwarnings('ignore')

session = sagemaker.Session()
bucket = session.default_bucket()
prefix = 'student-performance'
role = sagemaker.get_execution_role()

print(f'SageMaker Role: {role}')
print(f'S3 Bucket: {bucket}')
print(f'Region: {session.boto_region_name}')

## 1.1 Load Raw Data

In [ ]:
DATA_DIR = '../data/raw/'

df_mat = pd.read_csv(os.path.join(DATA_DIR, 'student-mat.csv'), sep=';')
df_por = pd.read_csv(os.path.join(DATA_DIR, 'student-por.csv'), sep=';')

print(f'Math dataset shape: {df_mat.shape}')
print(f'Portuguese dataset shape: {df_por.shape}')

## 1.2 Data Inspection

In [ ]:
print('=== Math Dataset Info ===')
df_mat.info()
print('\n=== Portuguese Dataset Info ===')
df_por.info()

In [ ]:
print('=== Math Dataset - First 5 Rows ===')
df_mat.head()

In [ ]:
print('=== Portuguese Dataset - First 5 Rows ===')
df_por.head()

In [ ]:
print('=== Math Dataset - Statistical Summary ===')
df_mat.describe()

In [ ]:
print('=== Portuguese Dataset - Statistical Summary ===')
df_por.describe()

## 1.3 Data Quality Checks

In [ ]:
print('=== Missing Values - Math ===')
print(df_mat.isnull().sum())
print(f'\nTotal missing: {df_mat.isnull().sum().sum()}')

print('\n=== Missing Values - Portuguese ===')
print(df_por.isnull().sum())
print(f'\nTotal missing: {df_por.isnull().sum().sum()}')

In [ ]:
print('=== Duplicate Rows ===')
print(f'Math duplicates: {df_mat.duplicated().sum()}')
print(f'Portuguese duplicates: {df_por.duplicated().sum()}')

In [ ]:
print('=== Data Types ===')
print(df_mat.dtypes.value_counts())

## 1.4 Combine Datasets
We combine Math and Portuguese datasets to increase sample size. A `subject` column tracks the source.

In [ ]:
df_mat['subject'] = 'Math'
df_por['subject'] = 'Portuguese'

df_combined = pd.concat([df_mat, df_por], axis=0, ignore_index=True)

print(f'Combined dataset shape: {df_combined.shape}')
print(f'\nSubject distribution:')
print(df_combined['subject'].value_counts())

## 1.5 Create Target Variable
Convert final grade (G3, scale 0-20) into binary classification:
- **High Performance**: G3 >= 12
- **Low Performance**: G3 < 12

In [ ]:
df_combined['performance'] = (df_combined['G3'] >= 12).astype(int)

print('Target Variable Distribution:')
print(df_combined['performance'].value_counts())
print(f'\nClass balance:')
print(df_combined['performance'].value_counts(normalize=True).round(3))

## 1.6 Save Combined Dataset

In [ ]:
output_path = '../data/processed/student_combined.csv'
df_combined.to_csv(output_path, index=False)
print(f'Combined dataset saved to: {output_path}')
print(f'Shape: {df_combined.shape}')

## 1.7 Upload to S3

In [ ]:
s3_raw_mat = session.upload_data(
    path='../data/raw/student-mat.csv',
    bucket=bucket,
    key_prefix=f'{prefix}/raw'
)
s3_raw_por = session.upload_data(
    path='../data/raw/student-por.csv',
    bucket=bucket,
    key_prefix=f'{prefix}/raw'
)
s3_processed = session.upload_data(
    path='../data/processed/student_combined.csv',
    bucket=bucket,
    key_prefix=f'{prefix}/processed'
)

print(f'Raw Math uploaded to: {s3_raw_mat}')
print(f'Raw Portuguese uploaded to: {s3_raw_por}')
print(f'Processed data uploaded to: {s3_processed}')

## Summary
- Loaded 2 datasets: Math (395 rows) and Portuguese (649 rows)
- Combined into single dataset (1044 rows, 35 columns)
- No missing values found
- Created binary target: `performance` (High=1, Low=0) based on G3 >= 12
- Data uploaded to S3 for SageMaker access